In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Install FAISS
!pip install -q faiss-cpu

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Creating knowledge base...")

kb = []
for idx, row in train.iterrows():
    correct_letter = row["answer"]          # A/B/C/D/E
    kb.append(str(row[correct_letter]))     # Store only correct answer text

print("Loading embedding model and creating index...")

model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = model.encode(kb, show_progress_bar=False)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.array(kb_embeddings).astype("float32"))

print("Knowledge base successfully created!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.4 MB/s eta 0:00:00:00:0100:01
Creating knowledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created!


In [3]:
# Zero-shot classifier
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Sample row for questions
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]


ans_150 = str(row_150[row_150["answer"]])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# Run zero-shot classification
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Display all labels with their scores
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label[:80]}... : {score:.3f}")

# Find the probability assigned to the correct answer
correct_score = dict(zip(result["labels"], result["scores"]))[ans_150]

print("\nGround Truth Answer:")
print(ans_150)
print(f"Probability = {correct_score:.3f}")

The butterfly effect is the phenomenon that a small change in the initial condit... : 0.384
The butterfly effect is the phenomenon that a large change in the initial condit... : 0.379
The butterfly effect is the phenomenon that a small change in the initial condit... : 0.093
The butterfly effect is the phenomenon that a small change in the initial condit... : 0.079
The butterfly effect is the phenomenon that a large change in the initial condit... : 0.066

Ground Truth Answer:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability = 0.384


In [5]:
# Embed the prompt
query_embedding = model.encode([prompt_150]).astype("float32")

# Retrieve top-10 nearest neighbors
k = 10
distances, indices = index.search(query_embedding, k)

print("Top-10 retrieved KB indices:")
for rank, idx in enumerate(indices[0], start=1):
    print(f"Rank {rank}: KB index = {idx}")

# Find the rank of the true document (KB index 150)
if 150 in indices[0]:
    rank = list(indices[0]).index(150) + 1
    print(f"\nTrue document found at Rank {rank}")
else:
    print("\nTrue document NOT found in the top 10.")

Top-10 retrieved KB indices:
Rank 1: KB index = 663
Rank 2: KB index = 1701
Rank 3: KB index = 1269
Rank 4: KB index = 1532
Rank 5: KB index = 576
Rank 6: KB index = 847
Rank 7: KB index = 1693
Rank 8: KB index = 1906
Rank 9: KB index = 168
Rank 10: KB index = 150

True document found at Rank 10


In [6]:
# Load Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Top-10 indices returned by FAISS
retrieved_indices = indices[0]

# Retrieve the corresponding documents
docs_10 = [kb[i] for i in retrieved_indices]

# Create (query, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Compute Cross-Encoder scores
ce_scores = cross_encoder.predict(pairs)

# Sort by score (highest first)
ranking = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Cross-Encoder Ranking:")
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    print(f"Rank {rank}: KB index={doc_idx}, Score={score:.4f}")

# Find the rank of the true document (KB index 150)
true_rank = None
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    if doc_idx == 150:
        true_rank = rank
        break

if true_rank is not None:
    print(f"\nTrue document is at Rank {true_rank}")
else:
    print("\nTrue document is not among the retrieved top-10 documents.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Ranking:
Rank 1: KB index=150, Score=4.7585
Rank 2: KB index=847, Score=4.7526
Rank 3: KB index=1693, Score=4.7526
Rank 4: KB index=1906, Score=4.7526
Rank 5: KB index=1269, Score=4.7375
Rank 6: KB index=1532, Score=4.7375
Rank 7: KB index=168, Score=4.7072
Rank 8: KB index=576, Score=4.6870
Rank 9: KB index=663, Score=4.6602
Rank 10: KB index=1701, Score=4.6602

True document is at Rank 1


In [7]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Prompt at row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode([prompt_42]).astype("float32")

# Retrieve top-5 documents
k = 5
distances, indices = index.search(query_embedding, k)

# Get retrieved documents
retrieved_docs = [kb[i] for i in indices[0]]

# Concatenate with a single space
concatenated_docs = " ".join(retrieved_docs)

# Build input string
text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Tokenize WITHOUT truncation
tokens = tokenizer(text, truncation=False)

# Count total tokens
num_tokens = len(tokens["input_ids"])

print("Total tokens:", num_tokens)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


In [8]:
# True row
row_150 = train.iloc[150]

# Prompt
prompt_150 = str(row_150["prompt"])

# True document from the KB
true_document = kb[150]

# Create the RAG prompt
rag_text = f"Context: {true_document} Question: {prompt_150}"

# Candidate labels
labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

# Ground-truth answer text
ground_truth = str(row_150[row_150["answer"]])

# Zero-shot classification
result = zs(
    rag_text,
    candidate_labels=labels_150,
    multi_label=False
)

# Display scores
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f}  {label}")

# Probability assigned to the correct option
correct_score = dict(zip(result["labels"], result["scores"]))[ground_truth]

print(f"\nGround-truth probability: {correct_score:.3f}")

0.989  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.004  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.003  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.002  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
0.002  The butterfly effect is the p

In [9]:
# Row 150
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ground_truth = str(row_150[row_150["answer"]])

# Use an unrelated document (KB index 999)
wrong_document = kb[999]

# Create the Adversarial RAG prompt
adv_rag = f"Context: {wrong_document} Question: {prompt_150}"

# Run zero-shot classification
result = zs(
    adv_rag,
    candidate_labels=labels_150,
    multi_label=False
)

# Show all scores
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f}  {label}")

# Probability assigned to the correct answer
correct_score = dict(zip(result["labels"], result["scores"]))[ground_truth]

print(f"\nProbability of correct option: {correct_score:.3f}")

0.529  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.425  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.020  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.019  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.007  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism c

In [10]:
# Compute Hit Rate@5 for first 100 rows

hits = 0
k = 5

for i in range(100):
    row = train.iloc[i]

    prompt = str(row["prompt"])

    # Ground-truth answer text
    correct_text = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode([prompt]).astype("float32")

    # Retrieve top-5 documents
    distances, indices = index.search(query_embedding, k)

    # Retrieved documents
    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check if the exact correct answer string appears in any retrieved document
    if any(correct_text == doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / 100 * 100

print(f"Hits: {hits}/100")
print(f"Hit Rate@5: {hit_rate:.1f}%")

Hits: 73/100
Hit Rate@5: 73.0%


In [11]:
from sklearn.metrics import average_precision_score

# Load Cross-Encoder once
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

k = 5
map3_scores = []

for i in range(20):

    row = train.iloc[i]
    prompt = str(row["prompt"])
    true_letter = row["answer"]

    # Candidate options
    option_texts = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"]),
    }

    # -------------------------
    # Stage 1: Retrieval
    # -------------------------
    query_embedding = model.encode([prompt]).astype("float32")
    distances, indices = index.search(query_embedding, k)

    docs = [kb[idx] for idx in indices[0]]

    # -------------------------
    # Stage 2: Reranking
    # -------------------------
    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    # -------------------------
    # Stage 3: Augmentation
    # -------------------------
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # -------------------------
    # Stage 4: Prediction
    # -------------------------
    candidate_labels = list(option_texts.values())

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels,
        multi_label=False,
    )

    # Map label text back to option letter
    text_to_letter = {v: k for k, v in option_texts.items()}

    ranked_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = ranked_letters[:3]

    # -------------------------
    # Stage 5: MAP@3
    # -------------------------
    ap = 0

    for rank, pred in enumerate(top3, start=1):
        if pred == true_letter:
            ap = 1 / rank
            break

    map3_scores.append(ap)

final_map3 = np.mean(map3_scores)

print("MAP@3 =", round(final_map3, 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MAP@3 = 0.975
